In [0]:
%sql
-- View will hold only those IATA_CODE whose respective columns will have null value
create or replace view aviation_project.bronze.airports_nulls as
select IATA_CODE from aviation_project.bronze.airports
where latitude is null or longitude is null or airport is null or city is null or state is null or country is null

In [0]:

df = spark.table('aviation_project.bronze.airports_nulls')

In [0]:
df.show()

In [0]:
# converting IATA_CODES into list of values 
iata_codes = [row['IATA_CODE'] for row in df.collect()]
print(iata_codes)

In [0]:
import requests
import time

"""
    Fetch airport details for a list of IATA codes from airport-data.com API.

    :param iata_codes: list of IATA codes (e.g. ["JFK", "LAX"])
    :param sleep_seconds: delay between API calls to avoid rate limits
    :return: list of airport records (dict)
 """

def fetch_airport_data(iata_codes, sleep_seconds=0.5):
 
    base_url = "https://airport-data.com/api/ap_info.json"
    #https://airport-data.com/api/ap_info.json?iata=DEF

    results = []

    for iata in iata_codes:
        try:
            response = requests.get(base_url, params={"iata": iata}, timeout=10)

            if response.status_code == 200:
                data = response.json()

                # API returns empty dict if airport not found
                if data:
                    data["iata_code"] = iata  # keep original code
                    results.append(data)
                else:
                    print(f"No data found for IATA: {iata}")
            else:
                print(f"Failed for {iata}, status: {response.status_code}")

        except Exception as e:
            print(f"Error fetching {iata}: {e}")

        time.sleep(sleep_seconds)  # rate-limit protection

    return results

 


In [0]:

airport_data = fetch_airport_data(iata_codes)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

api_schema = StructType([
    StructField("icao", StringType(), True),
    StructField("iata", StringType(), True),
    StructField("iata_code", StringType(), True),
    StructField("name", StringType(), True),
    StructField("location", StringType(), True),
    StructField("country", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("longitude", StringType(), True),
    StructField("latitude", StringType(), True),
    StructField("link", StringType(), True),
    StructField("status", IntegerType(), True)
])




In [0]:
#creating a df for all the airport details fetched from above API
api_df = spark.createDataFrame(airport_data, schema=api_schema)
api_df.show()

In [0]:
from pyspark.sql.functions import col, split, trim, when

#transforming the airport records fetched from API in the same format as airport table in bronze layer so that later we can merge that bronze layer airport table.

final_df = (
    api_df
    .withColumn("CITY", trim(split(col("location"), ",")[0]))
    .withColumn("STATE", trim(split(col("location"), ",")[1]))
    .withColumn("COUNTRY", trim(col("country")))
    .select(
        col("iata_code").alias("IATA_CODE"),
        col("name").alias("AIRPORT"),
        col("CITY"),
        col("STATE"),
        when(col("COUNTRY") == 'United States', 'USA').otherwise(col("COUNTRY")).alias("COUNTRY"),
        col("latitude").cast("double").alias("LATITUDE"),
        col("longitude").cast("double").alias("LONGITUDE")
    )
)
display(final_df)

In [0]:
final_df.write.mode("overwrite").format("delta").saveAsTable("aviation_project.bronze.airports_api")

In [0]:
%sql
select * from aviation_project.bronze.airports_api limit 3

In [0]:
%skip
create or replace table aviation_project.bronze.airports_copy as
select * from aviation_project.bronze.airports


In [0]:
 %skip

-- find the null values in all the columns

SELECT 
  SUM(CASE WHEN IATA_CODE IS NULL THEN 1 ELSE 0 END) AS IATA_nulls,
  SUM(CASE WHEN Airport IS NULL THEN 1 ELSE 0 END) AS Airport_nulls,
  SUM(CASE WHEN city IS NULL THEN 1 ELSE 0 END) AS city_nulls,
  SUM(CASE WHEN state IS NULL THEN 1 ELSE 0 END) AS state_nulls,
  SUM(CASE WHEN country IS NULL THEN 1 ELSE 0 END) AS country_nulls,
  SUM(CASE WHEN latitude IS NULL THEN 1 ELSE 0 END) AS latitude_nulls,
  SUM(CASE WHEN longitude IS NULL THEN 1 ELSE 0 END) AS longitude_nulls

FROM aviation_project.bronze.airports_copy;


In [0]:
%sql
MERGE INTO aviation_project.bronze.airports AS tgt
USING aviation_project.bronze.airports_api AS src
ON tgt.IATA_CODE = src.IATA_CODE

WHEN MATCHED AND (
tgt.latitude is null 
or tgt.longitude is null 
or tgt.airport is null 
or tgt.city is null 
or tgt.state is null 
or tgt.country is null
)
THEN UPDATE SET
    tgt.longitude = COALESCE(tgt.longitude, src.longitude),
    tgt.latitude = COALESCE(tgt.latitude, src.latitude),
    tgt.airport = COALESCE(tgt.airport, src.airport),
    tgt.city = COALESCE(tgt.city),
    tgt.state = COALESCE(tgt.state, src.state),
    tgt.country = COALESCE(tgt.country, src.country)



In [0]:
%sql
select * from aviation_project.bronze.airports
where IATA_CODE in ('ECP','UST','PBG')